# 面试问题：Node2Vec 怎样用 p/q 偏置随机游走和 Skip-gram 学习可用于链接预测的节点表示？

## 可直接复述的回答主线

1. Node2Vec 不直接遍历所有邻居，而是根据上一节点与候选节点的距离给二阶转移分配 1/p、1、1/q 权重。
2. 较大的 q 抑制向外探索，更像 BFS 捕捉社区同质性；较小 q 更像 DFS 捕捉结构角色。
3. 随机游走生成节点序列，再用窗口构造 center-context 正样本，并按 degree^0.75 采负样本。
4. Skip-gram 用两套 embedding，最大化正样本点积的 log-sigmoid、最小化负样本点积。
5. 评测应输出转移概率、真实 walks、样本对、loss、链接分数和 AUC，而不是只看二维可视化。
6. 生产还需有向/加权边、动态图更新、别名采样、分布式 walk、负采样去偏和冷启动特征融合。

后续实验会在同一批输入上依次展示朴素基线、手写核心机制、中间过程、失败修正和生产边界。

## 1. 真实案例与输入预览

案例是十个商品节点组成的同购图，电子与健身形成两个社区，中间只有一条弱桥。训练图含 13 条边，另外四条社区内未来同购边与四条跨社区负边用于链接排序评测。

In [1]:
import math  # 计算 embedding 余弦和 AUC。
import random  # 生成确定性 Node2Vec 随机游走。
import torch  # 使用基础 Embedding 和自动微分训练 Skip-gram。
torch.manual_seed(102)  # 固定 Skip-gram 初始化和负采样。
nodes = ["手机", "耳机", "充电器", "平板", "键盘", "瑜伽垫", "哑铃", "弹力带", "跑鞋", "水杯"]  # 定义十个具有社区语义的商品节点。
edges = [(0, 1), (1, 2), (2, 3), (3, 4), (4, 0), (1, 3), (5, 6), (6, 7), (7, 8), (8, 9), (9, 5), (6, 8), (4, 5)]  # 定义两个社区和一条跨社区桥的训练边。
positive_links = [(0, 2), (1, 4), (5, 7), (6, 9)]  # 定义未出现在训练图中的四条未来社区内同购边。
negative_links = [(0, 6), (1, 7), (2, 8), (3, 9)]  # 定义四条跨社区非同购候选。
adjacency = {node: [] for node in range(len(nodes))}  # 初始化手写无向邻接表。
for source, target in edges:  # 逐训练边写入双向邻接。
    adjacency[source].append(target)  # 加入正向邻居。
    adjacency[target].append(source)  # 加入反向邻居。
for node in adjacency:  # 固定每个节点候选顺序保证游走可复现。
    adjacency[node] = sorted(adjacency[node])  # 按节点编号排序邻居。
print("教学实验输入：商品同购图")  # 标记下方为离线图数据。
print("节点与邻居")  # 输出邻接表标题。
for node, name in enumerate(nodes):  # 逐节点展示商品和邻居。
    print(f"{node}:{name:<6} -> {[(neighbor, nodes[neighbor]) for neighbor in adjacency[node]]}")  # 输出当前商品的同购邻接。
print("holdout positives=", [(nodes[a], nodes[b]) for a, b in positive_links])  # 展示未来同购正例。
print("holdout negatives=", [(nodes[a], nodes[b]) for a, b in negative_links])  # 展示跨社区负例。

教学实验输入：商品同购图
节点与邻居
0:手机     -> [(1, '耳机'), (4, '键盘')]
1:耳机     -> [(0, '手机'), (2, '充电器'), (3, '平板')]
2:充电器    -> [(1, '耳机'), (3, '平板')]
3:平板     -> [(1, '耳机'), (2, '充电器'), (4, '键盘')]
4:键盘     -> [(0, '手机'), (3, '平板'), (5, '瑜伽垫')]
5:瑜伽垫    -> [(4, '键盘'), (6, '哑铃'), (9, '水杯')]
6:哑铃     -> [(5, '瑜伽垫'), (7, '弹力带'), (8, '跑鞋')]
7:弹力带    -> [(6, '哑铃'), (8, '跑鞋')]
8:跑鞋     -> [(6, '哑铃'), (7, '弹力带'), (9, '水杯')]
9:水杯     -> [(5, '瑜伽垫'), (8, '跑鞋')]
holdout positives= [('手机', '充电器'), ('耳机', '键盘'), ('瑜伽垫', '弹力带'), ('哑铃', '水杯')]
holdout negatives= [('手机', '哑铃'), ('耳机', '弹力带'), ('充电器', '跑鞋'), ('平板', '水杯')]


## 2. Baseline / 基线：用节点度乘积给候选链接打分

高度节点更容易被推荐，但度数不知道社区语义。正负候选的度乘积分布大量相同，排序 AUC 接近随机。

In [2]:
degrees = {node: len(neighbors) for node, neighbors in adjacency.items()}  # 计算十个节点训练度数。
def degree_score(link):  # 用两个端点度数乘积计算朴素链接分。
    source, target = link  # 解包候选边两个端点。
    return float(degrees[source] * degrees[target])  # 返回偏好热门节点的分数。
def pairwise_auc(positive_scores, negative_scores):  # 手写带 tie=0.5 的两两排序 AUC。
    comparisons = []  # 保存每个正负分数组合的胜负值。
    for positive in positive_scores:  # 遍历全部正例分数。
        for negative in negative_scores:  # 遍历全部负例分数。
            comparisons.append(1.0 if positive > negative else 0.5 if positive == negative else 0.0)  # 按严格胜出、平局或失败记分。
    return sum(comparisons) / len(comparisons)  # 返回正例排在负例前的概率。
baseline_positive_scores = [degree_score(link) for link in positive_links]  # 计算四个未来同购边的度分数。
baseline_negative_scores = [degree_score(link) for link in negative_links]  # 计算四个跨社区边的度分数。
baseline_auc = pairwise_auc(baseline_positive_scores, baseline_negative_scores)  # 计算度基线链接 AUC。
print("Baseline degree-product 链接分")  # 标记下表展示同分问题。
for label, links, scores in (("positive", positive_links, baseline_positive_scores), ("negative", negative_links, baseline_negative_scores)):  # 分别展示正负候选。
    for link, score in zip(links, scores):  # 逐候选边输出端点和分数。
        print(f"{label:<8} {nodes[link[0]]:<6}-{nodes[link[1]]:<6} score={score:.1f}")  # 输出当前候选的度乘积。
print(f"Baseline AUC={baseline_auc:.3f}")  # 展示热门度不能区分社区内链接。

Baseline degree-product 链接分
positive 手机    -充电器    score=4.0
positive 耳机    -键盘     score=9.0
positive 瑜伽垫   -弹力带    score=6.0
positive 哑铃    -水杯     score=6.0
negative 手机    -哑铃     score=6.0
negative 耳机    -弹力带    score=6.0
negative 充电器   -跑鞋     score=6.0
negative 平板    -水杯     score=6.0
Baseline AUC=0.500


## 3. 底层实现：二阶转移、窗口样本、负采样与 Skip-gram

对候选 x：返回上一节点权重 1/p；与上一节点相邻权重 1；否则权重 1/q。这里 p=1、q=2，偏向社区内 BFS。

In [3]:
def transition_probabilities(previous, current, p=1.0, q=2.0, invert_q=False):  # 计算 Node2Vec 二阶候选概率。
    candidates = adjacency[current]  # 读取当前节点全部邻居。
    weights = []  # 保存每个候选的未归一化权重。
    for candidate in candidates:  # 逐候选判断与上一节点的图距离类别。
        if candidate == previous:  # 检查是否立即返回上一节点。
            weight = 1.0 / p  # 应用 return 参数 p。
        elif candidate in adjacency[previous]:  # 检查候选是否也与上一节点相邻。
            weight = 1.0  # 对局部邻域候选使用单位权重。
        else:  # 其余候选会向外扩展游走。
            weight = q if invert_q else 1.0 / q  # 正确使用 1/q，并允许失败实验反向使用 q。
        weights.append(weight)  # 保存当前候选权重。
    total = sum(weights)  # 计算归一化分母。
    return candidates, [weight / total for weight in weights]  # 返回候选和概率。
def weighted_choice(candidates, probabilities, rng):  # 用累积分布手写离散采样。
    threshold = rng.random()  # 生成零到一确定性随机数。
    cumulative = 0.0  # 初始化累计概率。
    for candidate, probability in zip(candidates, probabilities):  # 顺序扫描候选概率。
        cumulative += probability  # 累加当前候选质量。
        if threshold <= cumulative:  # 检查随机数是否落入当前区间。
            return candidate  # 返回采中的节点。
    return candidates[-1]  # 处理浮点尾差并返回最后候选。
def node2vec_walk(start, length, rng, p=1.0, q=2.0):  # 从一个节点生成固定长度二阶随机游走。
    walk = [start]  # 用起始节点初始化序列。
    while len(walk) < length and adjacency[walk[-1]]:  # 在有邻居且未达到长度时继续。
        current = walk[-1]  # 读取当前节点。
        if len(walk) == 1:  # 第一跳没有上一节点可供二阶偏置。
            next_node = rng.choice(adjacency[current])  # 对第一跳邻居均匀采样。
        else:  # 第二跳开始使用 p/q 偏置。
            candidates, probabilities = transition_probabilities(walk[-2], current, p=p, q=q)  # 计算当前二阶转移分布。
            next_node = weighted_choice(candidates, probabilities, rng)  # 从累积分布采样下一节点。
        walk.append(next_node)  # 把采样节点加入序列。
    return walk  # 返回节点编号序列。
rng = random.Random(102)  # 创建固定 Python 随机数生成器。
walks = [node2vec_walk(start, 8, rng) for _ in range(10) for start in range(len(nodes))]  # 每个节点生成十条长度八的游走。
positive_pairs = []  # 保存 Skip-gram center-context 正样本。
window_size = 2  # 设定游走窗口半径为二。
for walk in walks:  # 逐游走展开窗口样本。
    for center_position, center in enumerate(walk):  # 逐位置作为 center。
        left = max(0, center_position - window_size)  # 计算窗口左边界。
        right = min(len(walk), center_position + window_size + 1)  # 计算窗口右开边界。
        for context_position in range(left, right):  # 遍历当前 center 周围节点。
            if context_position != center_position:  # 排除 center 自身。
                positive_pairs.append((center, walk[context_position]))  # 保存正样本节点对。
centers = torch.tensor([pair[0] for pair in positive_pairs], dtype=torch.long)  # 构造 center ID 张量。
contexts = torch.tensor([pair[1] for pair in positive_pairs], dtype=torch.long)  # 构造 context ID 张量。
negative_weights = torch.tensor([degrees[node] ** 0.75 for node in range(len(nodes))], dtype=torch.float32)  # 构造 degree^0.75 负采样分布。
negative_generator = torch.Generator().manual_seed(102)  # 固定负采样生成器。
negatives = torch.multinomial(negative_weights, len(positive_pairs) * 4, replacement=True, generator=negative_generator).reshape(len(positive_pairs), 4)  # 为每个正样本采四个负节点。
class SkipGram(torch.nn.Module):  # 定义显式 forward 的双 embedding Skip-gram。
    def __init__(self, node_count, embedding_dim=10):  # 初始化 center 和 context 参数。
        super().__init__()  # 注册 PyTorch 参数管理。
        self.input_embedding = torch.nn.Embedding(node_count, embedding_dim)  # 定义中心节点向量表。
        self.output_embedding = torch.nn.Embedding(node_count, embedding_dim)  # 定义上下文节点向量表。
    def forward(self, center_ids, context_ids, negative_ids):  # 计算正负点积的 Skip-gram 损失。
        center_vectors = self.input_embedding(center_ids)  # 查找批次 center 向量。
        context_vectors = self.output_embedding(context_ids)  # 查找正 context 向量。
        negative_vectors = self.output_embedding(negative_ids)  # 查找每个 center 的四个负向量。
        positive_logits = (center_vectors * context_vectors).sum(dim=1)  # 计算正样本点积。
        negative_logits = torch.einsum("bd,bkd->bk", center_vectors, negative_vectors)  # 计算四个负样本点积。
        positive_loss = -torch.nn.functional.logsigmoid(positive_logits)  # 最大化正点积概率。
        negative_loss = -torch.nn.functional.logsigmoid(-negative_logits).sum(dim=1)  # 最小化负点积并求和。
        return (positive_loss + negative_loss).mean(), positive_logits, negative_logits  # 返回平均损失和中间 logits。
model = SkipGram(len(nodes))  # 创建十节点 Skip-gram 模型。
history = []  # 保存真实 backward 训练轨迹。
for step in range(180):  # 对固定 walk 样本执行全批次训练。
    model.zero_grad(set_to_none=True)  # 清除上一步梯度。
    loss, positive_logits, negative_logits = model(centers, contexts, negatives)  # 前向计算正负采样损失。
    loss.backward()  # 对两套 embedding 执行真实反向传播。
    gradient_norm = math.sqrt(sum(float((parameter.grad ** 2).sum().item()) for parameter in model.parameters()))  # 汇总 embedding 梯度范数。
    with torch.no_grad():  # 在无梯度上下文手动执行 SGD。
        for parameter in model.parameters():  # 遍历输入与输出 embedding。
            parameter.add_(parameter.grad, alpha=-0.08)  # 使用固定学习率更新节点向量。
    if step % 45 == 0 or step == 179:  # 每四十五步记录一次训练状态。
        history.append({"step": step, "loss": loss.item(), "positive_logit": positive_logits.mean().item(), "negative_logit": negative_logits.mean().item(), "gradient_norm": gradient_norm})  # 保存损失和正负点积。
print("随机游走样例：", [[nodes[node] for node in walk] for walk in walks[:6]])  # 展示真实 p/q 采样序列。
print("前十个Skip-gram pair：", [(nodes[center], nodes[context]) for center, context in positive_pairs[:10]])  # 展示窗口如何产生训练样本。
print("训练轨迹：", history)  # 展示损失、点积和梯度变化。

随机游走样例： [['手机', '耳机', '充电器', '耳机', '充电器', '平板', '耳机', '充电器'], ['耳机', '充电器', '平板', '耳机', '充电器', '平板', '充电器', '平板'], ['充电器', '平板', '充电器', '平板', '耳机', '手机', '耳机', '充电器'], ['平板', '键盘', '瑜伽垫', '哑铃', '弹力带', '哑铃', '弹力带', '哑铃'], ['键盘', '手机', '耳机', '手机', '耳机', '充电器', '平板', '充电器'], ['瑜伽垫', '哑铃', '跑鞋', '哑铃', '跑鞋', '哑铃', '瑜伽垫', '键盘']]
前十个Skip-gram pair： [('手机', '耳机'), ('手机', '充电器'), ('耳机', '手机'), ('耳机', '充电器'), ('耳机', '耳机'), ('充电器', '手机'), ('充电器', '耳机'), ('充电器', '耳机'), ('充电器', '充电器'), ('耳机', '耳机')]
训练轨迹： [{'step': 0, 'loss': 6.757551193237305, 'positive_logit': -0.8022870421409607, 'negative_logit': -0.17677976191043854, 'gradient_norm': 1.3045630481320225}, {'step': 45, 'loss': 3.5069332122802734, 'positive_logit': -0.6991150975227356, 'negative_logit': -0.759078860282898, 'gradient_norm': 0.6219623178493404}, {'step': 90, 'loss': 2.7169623374938965, 'positive_logit': -0.7711434960365295, 'negative_logit': -1.1361137628555298, 'gradient_norm': 0.34590401712286756}, {'step': 135, 'loss': 2.4205222

## 4. 链接预测结果与结果解读

无向图中 center/context 角色对称，因此把训练后的 input 与 output embedding 相加，再用 cosine 给同一组 holdout 正负边打分，并与 degree-product AUC 对照。

In [4]:
learned_embeddings = (model.input_embedding.weight + model.output_embedding.weight).detach()  # 合并 center 与 context 向量供无向链接打分。
learned_embeddings = learned_embeddings / learned_embeddings.norm(dim=1, keepdim=True).clamp_min(1.0e-8)  # 逐节点归一化供 cosine 打分。
def embedding_score(link):  # 计算候选链接两个节点的 embedding cosine。
    source, target = link  # 解包候选端点。
    return float((learned_embeddings[source] @ learned_embeddings[target]).item())  # 返回单位向量点积。
learned_positive_scores = [embedding_score(link) for link in positive_links]  # 计算四个未来同购边表示分数。
learned_negative_scores = [embedding_score(link) for link in negative_links]  # 计算四个跨社区边表示分数。
learned_auc = pairwise_auc(learned_positive_scores, learned_negative_scores)  # 计算 Node2Vec 链接 AUC。
print("link                  label  degree_score  embedding_cos")  # 输出逐候选链接对照表头。
for label, links in ((1, positive_links), (0, negative_links)):  # 依次展示正负候选。
    for link in links:  # 逐边比较度数和 embedding。
        print(f"{nodes[link[0]]:<6}-{nodes[link[1]]:<6} {label:>5} {degree_score(link):>13.2f} {embedding_score(link):>14.4f}")  # 输出当前链接两种分数。
print(f"结果解读：degree-product AUC={baseline_auc:.3f}，Node2Vec Skip-gram AUC={learned_auc:.3f}；提升来自社区游走共现。")  # 解释受控链接排序收益。

link                  label  degree_score  embedding_cos
手机    -充电器        1          4.00         0.1731
耳机    -键盘         1          9.00        -0.0302
瑜伽垫   -弹力带        1          6.00         0.0721
哑铃    -水杯         1          6.00         0.1840
手机    -哑铃         0          6.00        -0.1175
耳机    -弹力带        0          6.00        -0.3738
充电器   -跑鞋         0          6.00         0.0736
平板    -水杯         0          6.00         0.1132
结果解读：degree-product AUC=0.500，Node2Vec Skip-gram AUC=0.750；提升来自社区游走共现。


## 5. 失败案例与修正：把向外权重 1/q 错写成 q

从“手机→耳机”继续游走时，候选“手机”是 return，候选“充电器/平板”是 outward。q=2 本应抑制 outward；错写成 q 会反而提高向外概率。

In [5]:
correct_candidates, correct_probabilities = transition_probabilities(previous=0, current=1, p=1.0, q=2.0, invert_q=False)  # 计算正确 1/q 二阶分布。
wrong_candidates, wrong_probabilities = transition_probabilities(previous=0, current=1, p=1.0, q=2.0, invert_q=True)  # 计算错误 q 二阶分布。
correct_distribution = {nodes[node]: probability for node, probability in zip(correct_candidates, correct_probabilities)}  # 映射正确候选商品概率。
wrong_distribution = {nodes[node]: probability for node, probability in zip(wrong_candidates, wrong_probabilities)}  # 映射错误候选商品概率。
correct_return_probability = correct_distribution["手机"]  # 读取正确实现立即返回概率。
wrong_return_probability = wrong_distribution["手机"]  # 读取错误实现立即返回概率。
print(f"错误行为：把1/q写成q，distribution={wrong_distribution}，return_probability={wrong_return_probability:.3f}")  # 展示 q 语义被反转。
print(f"修正行为：使用1/q，distribution={correct_distribution}，return_probability={correct_return_probability:.3f}")  # 展示正确 BFS 偏置。

错误行为：把1/q写成q，distribution={'手机': 0.2, '充电器': 0.4, '平板': 0.4}，return_probability=0.200
修正行为：使用1/q，distribution={'手机': 0.5, '充电器': 0.25, '平板': 0.25}，return_probability=0.500


## 6. 生产边界

十节点图会被向量记忆。生产需要按边权/时间采样、alias table 加速二阶转移、分布式 walk、动态增量训练、负采样去除真邻居、孤立和新节点特征融合，并用时间切分链接预测避免未来边泄漏。

In [6]:
node2vec_diagnostics = {"nodes": len(nodes), "training_edges": len(edges), "walks": len(walks), "positive_pairs": len(positive_pairs), "final_loss": history[-1]["loss"], "baseline_auc": baseline_auc, "node2vec_auc": learned_auc, "correct_return_probability": correct_return_probability, "wrong_return_probability": wrong_return_probability}  # 汇总图、训练、评测和失败指标。
print("生产监控快照：", node2vec_diagnostics)  # 输出 Node2Vec 管线应持续观察的信号。

生产监控快照： {'nodes': 10, 'training_edges': 13, 'walks': 100, 'positive_pairs': 2600, 'final_loss': 2.271160125732422, 'baseline_auc': 0.5, 'node2vec_auc': 0.75, 'correct_return_probability': 0.5, 'wrong_return_probability': 0.2}


## 7. 最小回归测试

断言覆盖图规模、真实训练、游走样本、链接排序和 p/q 失败修正。

In [7]:
assert len(nodes) >= 6 and len(edges) >= 10 and len(walks) >= len(nodes)  # 保证案例具有非平凡图和足够游走。
assert history[-1]["loss"] < history[0]["loss"] and all(row["gradient_norm"] > 0.0 for row in history)  # 保证 Skip-gram 实际 forward/backward 学习。
assert len(positive_pairs) > 100 and centers.shape[0] == negatives.shape[0]  # 保证窗口样本和负采样维度一致。
assert learned_auc > baseline_auc and learned_auc >= 0.75  # 保证同一 holdout 边上表示优于度数基线。
assert abs(sum(correct_probabilities) - 1.0) < 1.0e-12 and abs(sum(wrong_probabilities) - 1.0) < 1.0e-12  # 保证两种转移分布均正确归一化。
assert correct_return_probability > wrong_return_probability  # 保证 q=2 的正确实现抑制 outward 而错误实现相反。